# CryptoHack: Diffie-Hellman

Solutions to the **Diffie-Hellman** course on [CryptoHack](https://cryptohack.org/courses/diffie-hellman/course_details/).

In Diffie-Hellman key exchange, Alice and Bob agree on a prime `p` and a generator `g`.
Alice picks a secret `a` and sends `A = g^a mod p`; Bob picks `b` and sends `B = g^b mod p`.
Both can then compute the shared secret `g^(ab) = B^a = A^b mod p`.

# Starter

## Working with Fields

In the finite field `F_p`, every non-zero element has a multiplicative inverse.
Find `d = 209⁻¹ mod 991`.

In [1]:
print(pow(209, -1, 991))

569


## Generators of Groups

`g` generates `F_p*` if its order is exactly `p - 1`. By Lagrange's theorem, the order of `g` divides `p - 1`,
so it suffices to check that `g^((p-1)/q) ≠ 1` for every prime factor `q` of `p - 1`.
The flag is the smallest generator.

In [2]:
from sympy import primefactors


def is_generator(g, p):
    return all(pow(g, (p - 1) // q, p) != 1 for q in primefactors(p - 1))


p = 28151
print(next(g for g in range(2, p) if is_generator(g, p)))

7


## Computing Public Values

Alice's public value is `A = g^a mod p`.

In [3]:
g = 2
p = 2410312426921032588552076022197566074856950548502459942654116941958108831682612228890093858261341614673227141477904012196503648957050582631942730706805009223062734745341073406696246014589361659774041027169249453200378729434170325843778659198143763193776859869524088940195577346119843545301547043747207749969763750084308926339295559968882457872412993810129130294592999947926365264059284647209730384947211681434464714438488520940127459844288859336526896320919633919
a = 972107443837033796245864316200458246846904598488981605856765890478853088246897345487328491037710219222038930943365848626194109830309179393018216763327572120124760140018038673999837643377590434413866611132403979547150659053897355593394492586978400044375465657296027592948349589216415363722668361328689588996541370097559090335137676411595949335857341797148926151694299575970292809805314431447043469447485957669949989090202320234337890323293401862304986599884732815
print(pow(g, a, p))

1806857697840726523322586721820911358489420128129248078673933653533930681676181753849411715714173604352323556558783759252661061186320274214883104886050164368129191719707402291577330485499513522368289395359523901406138025022522412429238971591272160519144672389532393673832265070057319485399793101182682177465364396277424717543434017666343807276970864475830391776403957550678362368319776566025118492062196941451265638054400177248572271342548616103967411990437357924


## Computing Shared Secrets

We play Bob: knowing Alice's public value `A` and Bob's secret `b`, the shared secret is `A^b mod p`.
Alice obtains the same value by computing `B^a mod p`.

In [4]:
g = 2
p = 2410312426921032588552076022197566074856950548502459942654116941958108831682612228890093858261341614673227141477904012196503648957050582631942730706805009223062734745341073406696246014589361659774041027169249453200378729434170325843778659198143763193776859869524088940195577346119843545301547043747207749969763750084308926339295559968882457872412993810129130294592999947926365264059284647209730384947211681434464714438488520940127459844288859336526896320919633919
A = 70249943217595468278554541264975482909289174351516133994495821400710625291840101960595720462672604202133493023241393916394629829526272643847352371534839862030410331485087487331809285533195024369287293217083414424096866925845838641840923193480821332056735592483730921055532222505605661664236182285229504265881752580410194731633895345823963910901731715743835775619780738974844840425579683385344491015955892106904647602049559477279345982530488299847663103078045601
b = 12019233252903990344598522535774963020395770409445296724034378433497976840167805970589960962221948290951873387728102115996831454482299243226839490999713763440412177965861508773420532266484619126710566414914227560103715336696193210379850575047730388378348266180934946139100479831339835896583443691529372703954589071507717917136906770122077739814262298488662138085608736103418601750861698417340264213867753834679359191427098195887112064503104510489610448294420720
B = 518386956790041579928056815914221837599234551655144585133414727838977145777213383018096662516814302583841858901021822273505120728451788412967971809038854090670743265187138208169355155411883063541881209288967735684152473260687799664130956969450297407027926009182761627800181901721840557870828019840218548188487260441829333603432714023447029942863076979487889569452186257333512355724725941390498966546682790608125613166744820307691068563387354936732643569654017172
print(pow(A, b, p))

1174130740413820656533832746034841985877302086316388380165984436672307692443711310285014138545204369495478725102882673427892104539120952393788961051992901649694063179853598311473820341215879965343136351436410522850717408445802043003164658348006577408558693502220285700893404674592567626297571222027902631157072143330043118418467094237965591198440803970726604537807146703763571606861448354607502654664700390453794493176794678917352634029713320615865940720837909466


## Deriving Symmetric Keys

A shared secret is a large integer, not a symmetric key. Here the AES key is derived by hashing the secret
with SHA-1 and keeping the first 16 bytes, then the flag is decrypted with AES-CBC.

The `decrypt_flag` function is reused by the following challenges.

In [5]:
import hashlib

from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad


def decrypt_flag(shared_secret: int, iv: str, ciphertext: str) -> str:
    """Derive an AES key from a Diffie-Hellman shared secret and decrypt the flag."""
    key = hashlib.sha1(str(shared_secret).encode("ascii")).digest()[:16]
    cipher = AES.new(key, AES.MODE_CBC, bytes.fromhex(iv))
    plaintext = cipher.decrypt(bytes.fromhex(ciphertext))
    try:
        plaintext = unpad(plaintext, AES.block_size)
    except ValueError:  # the servers do not pad a flag that already fills whole blocks
        pass
    return plaintext.decode("ascii")


p = 2410312426921032588552076022197566074856950548502459942654116941958108831682612228890093858261341614673227141477904012196503648957050582631942730706805009223062734745341073406696246014589361659774041027169249453200378729434170325843778659198143763193776859869524088940195577346119843545301547043747207749969763750084308926339295559968882457872412993810129130294592999947926365264059284647209730384947211681434464714438488520940127459844288859336526896320919633919
A = 112218739139542908880564359534373424013016249772931962692237907571990334483528877513809272625610512061159061737608547288558662879685086684299624481742865016924065000555267977830144740364467977206555914781236397216033805882207640219686011643468275165718132888489024688846101943642459655423609111976363316080620471928236879737944217503462265615774774318986375878440978819238346077908864116156831874695817477772477121232820827728424890845769152726027520772901423784
b = 197395083814907028991785772714920885908249341925650951555219049411298436217190605190824934787336279228785809783531814507661385111220639329358048196339626065676869119737979175531770768861808581110311903548567424039264485661330995221907803300824165469977099494284722831845653985392791480264712091293580274947132480402319812110462641143884577706335859190668240694680261160210609506891842793868297672619625924001403035676872189455767944077542198064499486164431451944
iv = "737561146ff8194f45290f5766ed6aba"
ciphertext = "39c99bf2f0c14678d6a5416faef954b5893c316fc3c48622ba1fd6a9fe85f3dc72a29c394cf4bc8aff6a7b21cae8e12c"

shared_secret = pow(A, b, p)
print(decrypt_flag(shared_secret, iv, ciphertext))

crypto{sh4r1ng_s3cret5_w1th_fr13nd5}


# Man In The Middle

The next challenges place us between Alice and Bob: we receive every message they exchange and choose what to forward.
The server prefixes each JSON message with a short text such as `Intercepted from Alice: `.

In [6]:
import json

from pwn import context, remote

context.log_level = "error"


def recv_json(conn):
    """Read one line from the server and parse the JSON object it contains."""
    line = conn.recvline().decode()
    return json.loads(line[line.index("{"):])


def send_json(conn, obj):
    conn.sendline(json.dumps(obj).encode())

## Parameter Injection

Since nothing authenticates the exchanged values, we replace both public values with `1`.
Each party then computes the shared secret `1^secret mod p = 1`, which we know.

In [7]:
with remote("socket.cryptohack.org", 13371) as conn:
    alice = recv_json(conn)                                         # {p, g, A} from Alice
    send_json(conn, {"p": alice["p"], "g": alice["g"], "A": "0x01"})  # Bob computes 1^b = 1

    recv_json(conn)                                                 # {B} from Bob
    send_json(conn, {"B": "0x01"})                                  # Alice computes 1^a = 1

    encrypted = recv_json(conn)                                     # {iv, encrypted_flag} from Alice

print(decrypt_flag(1, encrypted["iv"], encrypted["encrypted_flag"]))

crypto{n1c3_0n3_m4ll0ry!!!!!!!!}


## Export-grade

Alice and Bob negotiate the group size, and we can tamper with the negotiation (as in the Logjam attack).
We force the weakest option, a 64-bit prime, for which the discrete logarithm problem is easy:
`sympy.discrete_log` recovers Alice's secret `a` from `A = g^a mod p` with the Pohlig-Hellman algorithm.

In [8]:
from sympy.ntheory import discrete_log

with remote("socket.cryptohack.org", 13379) as conn:
    supported = recv_json(conn)
    print("Alice supports:", supported["supported"])
    send_json(conn, {"supported": ["DH64"]})  # downgrade Alice's offer

    chosen = recv_json(conn)
    send_json(conn, chosen)                   # Bob picks DH64

    alice = recv_json(conn)
    send_json(conn, alice)
    bob = recv_json(conn)
    send_json(conn, bob)
    encrypted = recv_json(conn)

p, g, A, B = (int(x, 16) for x in (alice["p"], alice["g"], alice["A"], bob["B"]))

a = discrete_log(p, A, g)
assert pow(g, a, p) == A
print("Alice's secret:", a)

shared_secret = pow(B, a, p)
print(decrypt_flag(shared_secret, encrypted["iv"], encrypted["encrypted_flag"]))

Alice supports: ['DH1536', 'DH1024', 'DH512', 'DH256', 'DH128', 'DH64']


Alice's secret: 1007039862023459824
crypto{d0wn6r4d35_4r3_d4n63r0u5}
